## Dumping and Restoring the Environment

Every LAILA policy carries configuration across communication,
command (taskforces), memory (pools), and identity.  The
`laila.environment` attribute captures all of this as a plain
JSON-serializable dictionary — the same structure that
`laila.args` accepts.

### What this notebook does
1. Create a policy with **custom communication settings**.
2. Dump the live environment to a JSON file.
3. Inspect the dump.
4. Load the dump back into `laila.args` and rebuild a policy.
5. Verify the rebuilt policy matches the original.

In [ ]:
import json
import laila
from laila.macros.defaults import DefaultPolicy
from dotmap import DotMap

### 1 — Create a policy with custom settings

In [ ]:
policy = DefaultPolicy()

policy.central.communication.host = "192.168.1.50"
policy.central.communication.port = 9090
policy.central.communication.peer_secret_key = "my-secret-key"

laila.active_policy = policy

print(f"Policy ID: {policy.global_id}")
print(f"Host:      {policy.central.communication.host}")
print(f"Port:      {policy.central.communication.port}")
print(f"Secret:    {policy.central.communication.peer_secret_key}")

### 2 — Dump the environment

In [ ]:
env = laila.environment

print(json.dumps(env, indent=2, default=str))

### 3 — Save to a JSON file

In [ ]:
with open("env_snapshot.json", "w") as f:
    json.dump(env, f, indent=2, default=str)

print("Saved to env_snapshot.json")

### 4 — Restore from the snapshot

Load the JSON file into `laila.args` and create a fresh policy.
Fields with fixed args paths (communication host, port, secret)
are restored exactly.

In [ ]:
with open("env_snapshot.json") as f:
    loaded = json.load(f)

laila.args = DotMap(loaded)

restored = DefaultPolicy()
laila.active_policy = restored

print(f"Restored host:   {restored.central.communication.host}")
print(f"Restored port:   {restored.central.communication.port}")
print(f"Restored secret: {restored.central.communication.peer_secret_key}")

### 5 — Verify the match

In [ ]:
assert restored.central.communication.host == "192.168.1.50"
assert restored.central.communication.port == 9090
assert restored.central.communication.peer_secret_key == "my-secret-key"

print("All fields restored correctly.")

### 6 — Compare the two environment dumps

In [ ]:
env_restored = laila.environment

comm_original = env["policy"]["central"]["communication"]
comm_restored = env_restored["policy"]["central"]["communication"]

print("Original comm: ", json.dumps(comm_original, default=str))
print("Restored comm: ", json.dumps(comm_restored, default=str))

assert comm_original == comm_restored
print("\nCommunication config is identical.")

### Cleanup

In [ ]:
import os
os.remove("env_snapshot.json")
print("Cleaned up.")